# 01 · Weeks 1-2: data intake (REAL dataset)
RetailRocket treated as a new customer's first data drop. The checks live in `sql/01_intake_checks.sql` and are run by `python src/run_intake.py` (needs the raw files). This notebook reads the **published aggregates** so it runs from a clean clone.

In [1]:
from pathlib import Path
import json
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
pd.set_option('display.width', 160); pd.set_option('display.max_colwidth', 80)

## The checks, graded OK / WARNING / BLOCKER

In [2]:
checks = pd.read_csv(ROOT / 'data/public/intake/checks.csv')
checks[['status', 'label', 'affected', 'rate', 'detail']]

,status,label,affected,rate,detail
0,OK,Required columns present,0,0.000000,"user_id, event_type, item_id, timestamp + catalog.item_id"
1,OK,Timestamp type and time zone,0,0.000000,column type BIGINT; Unix time in ms: UTC by definition
2,OK,Timestamps that cannot be parsed,0,0.000000,NaN
3,OK,"Rows missing user, item, type or timestamp",0,0.000000,NaN
4,OK,Timestamps outside 2000-01-01 .. now,0,0.000000,range 2015-05-03 03:00:04.384 -> 2015-09-18 02:59:47.788 (UTC)
5,OK,Event types outside the agreed list,0,0.000000,"view=2664312, addtocart=69332, transaction=22457"
6,WARNING,Events on items missing from the catalog,255585,0.092734,49815 distinct item ids missing from the catalog
7,WARNING,Catalog items with no event in the period,231807,0.555821,NaN
8,OK,Duplicate item ids in the catalog,0,0.000000,NaN
9,OK,Exact duplicate events,460,0.000167,NaN


## Filtering rules and how many rows each one removes

In [3]:
pd.read_csv(ROOT / 'data/public/intake/filtering.csv')

,step,rows_after,removed,share_removed
0,raw events,2756101,0,0.000000
1,invalid rows,2756101,0,0.000000
2,exact duplicates,2755641,460,0.000167
3,repeats < 1 s,2750936,4705,0.001707
4,bot users (rules 1-2),2657575,93361,0.033874


## One check in SQL: near-duplicates (same user, item and type less than 1 s apart)
`lag()` looks at the previous event of the same user / item / type; a gap between 0 and 1 second is a double tap or a reload.

In [4]:
sql = (ROOT / 'sql/01_intake_checks.sql').read_text()
print(sql[sql.index('-- name: check_near_duplicates'):sql.index('-- name: check_missing_days')])

-- name: check_near_duplicates
-- Same user, item and event type less than 1 second after the previous one
-- (double taps, page reloads). Exact duplicates (0 s) are counted above.
WITH ordered AS (
    SELECT ts - lag(ts) OVER (PARTITION BY user_id, item_id, event_type ORDER BY ts) AS gap
    FROM events_raw
)
SELECT count(*) FILTER (WHERE gap > INTERVAL 0 SECOND AND gap < INTERVAL 1 SECOND) AS affected,
       count(*)                                                                     AS base,
       NULL                                                                         AS detail
FROM ordered;





## Daily volume: the last two days are partial

In [5]:
daily = pd.read_csv(ROOT / 'data/public/intake/daily_volume.csv')
daily.tail(4)

,day,raw_events,clean_events
135,2015-09-15,20427,19817
136,2015-09-16,11495,10784
137,2015-09-17,10128,9750
138,2015-09-18,1528,1467
